# 📜 Data Preparation — Demo & Results
## โครงการวิเคราะห์รัฐธรรมนูญไทย 20 ฉบับ (CPE232 Data Models)

---

| รายละเอียด | ข้อมูล |
|-----------|--------|
| **วิชา** | CPE232 Data Models |
| **โปรเจกต์** | Twenty Constitutions Analysis |
| **ส่วนงาน** | Data Preparation (OCR + Text Extraction) |
| **แหล่งข้อมูล** | ระบบคลังสารสนเทศรัฐสภา |
| **เครื่องมือ OCR** | Typhoon OCR 1.5 (typhoon-ai/typhoon-ocr1.5-2b) |
| **เครื่องมือ Extraction** | PyMuPDF / pdfplumber |

**สมาชิกกลุ่ม:** กันต์ธีร์ ดวงมณี · นัธทวัฒน์ ปริมสิริคุณาวุฒิ · วิศิษฐ์ สุวรรณเนาว์ · ศุภวิชญ์ มารยาท · พลวริษฐ์ วัฒนเหมรัตน์

---

> ⚠️ **Notebook นี้ใช้สำหรับแสดงผล (Demo) เท่านั้น**  
> ไม่มีการรัน OCR หรือ Text Extraction จริงใน Notebook นี้  
> สำหรับรัน Pipeline จริง ให้ใช้ `python run_pipeline.py`


## 🔄 Pipeline Overview

```
┌─────────────────────────────────────────────────────────┐
│           INPUT: PDF Files (38 ไฟล์)                   │
└─────────────────────────────────────────────────────────┘
                       │
                       ▼
         ┌─────────────────────────┐
         │  STEP 0: Classification │  ← config.py
         └─────────────────────────┘
              │               │
     Image PDF (12 ฉบับ)   Text PDF (26 ฉบับ)
     พ.ศ. 2475–2502         พ.ศ. 2511–2564
              │               │
              ▼               ▼
    ┌──────────────┐  ┌──────────────────┐
    │ Typhoon OCR  │  │  PyMuPDF /       │
    │    1.5       │  │  pdfplumber      │
    └──────┬───────┘  └────────┬─────────┘
           │                  │
           └────────┬─────────┘
                    ▼
       ┌────────────────────────┐
       │  Post-Processing       │
       │  normalize / clean     │
       └────────────┬───────────┘
                    ▼
       ┌────────────────────────┐
       │  Validation & QA       │
       └────────────┬───────────┘
                    ▼
       ┌────────────────────────┐
       │  JSON + CSV + TXT      │  ← data/processed/
       └────────────────────────┘
```


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1: Setup — imports, paths, style
# ═══════════════════════════════════════════════════════════
import os
import json
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from IPython.display import display, HTML

warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────
NOTEBOOK_DIR  = Path('.').resolve()
PROCESSED_DIR = NOTEBOOK_DIR / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# ── Matplotlib Thai font ──────────────────────────────────
plt.rcParams['font.family']        = ['TH Sarabun New', 'Tahoma', 'DejaVu Sans']
plt.rcParams['font.size']          = 11
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi']         = 110
plt.rcParams['axes.spines.top']    = False
plt.rcParams['axes.spines.right']  = False

# ── Color palette ─────────────────────────────────────────
ERA_COLORS = {
    'early_democracy':   '#4CAF50',
    'post_coup_1947':    '#FF9800',
    'dictatorship':      '#f44336',
    'democratic_spring': '#2196F3',
    'semi_democracy':    '#9C27B0',
    'modern_democracy':  '#009688',
    'post_coup_2006':    '#FF5722',
    'post_coup_2014':    '#795548',
}

ERA_LABELS_TH = {
    'early_democracy':   'ยุคประชาธิปไตยแรกเริ่ม (2475–2490)',
    'post_coup_1947':    'ยุคหลังรัฐประหาร 2490 (2490–2502)',
    'dictatorship':      'ยุคเผด็จการทหาร (2502–2516)',
    'democratic_spring': 'ยุคประชาธิปไตย (2517–2519)',
    'semi_democracy':    'ยุคกึ่งประชาธิปไตย (2520–2533)',
    'modern_democracy':  'ยุคประชาธิปไตยสมัยใหม่ (2534–2549)',
    'post_coup_2006':    'ยุคหลังรัฐประหาร 2549 (2549–2557)',
    'post_coup_2014':    'ยุคหลังรัฐประหาร 2557 (2557–ปัจจุบัน)',
}

print('✅ Setup เสร็จสิ้น')
print(f'📁 PROCESSED_DIR : {PROCESSED_DIR}')
print(f'📄 ไฟล์ที่ประมวลผลแล้ว : {len(list(PROCESSED_DIR.glob("const_*.json")))} ไฟล์')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2: Mock Data + Load Real Data
# ถ้า Pipeline ยังไม่ได้รัน → ใช้ MOCK_DATA เป็น fallback
# ═══════════════════════════════════════════════════════════

MOCK_DATA = [
    # ── Image PDFs (ต้องทำ OCR) ────────────────────────────
    {"id": "const_2475", "year_th": 2475, "year_ce": 1932,
     "name_short": "รธน. 2475",
     "source_type": "image_pdf", "processing_method": "typhoon-ocr-1.5",
     "era": "early_democracy",   "regime_type": "civilian",
     "total_pages": 14, "total_chars": 19000, "total_words_approx": 4050,
     "quality_score": 85},
    {"id": "const_2482", "year_th": 2482, "year_ce": 1939,
     "name_short": "รธน. 2482",
     "source_type": "image_pdf", "processing_method": "typhoon-ocr-1.5",
     "era": "early_democracy",   "regime_type": "civilian",
     "total_pages":  3, "total_chars":  1800, "total_words_approx":  380,
     "quality_score": 82},
    {"id": "const_2483", "year_th": 2483, "year_ce": 1940,
     "name_short": "รธน. 2483",
     "source_type": "image_pdf", "processing_method": "typhoon-ocr-1.5",
     "era": "early_democracy",   "regime_type": "civilian",
     "total_pages":  2, "total_chars":  1400, "total_words_approx":  310,
     "quality_score": 80},
    {"id": "const_2485", "year_th": 2485, "year_ce": 1942,
     "name_short": "รธน. 2485",
     "source_type": "image_pdf", "processing_method": "typhoon-ocr-1.5",
     "era": "early_democracy",   "regime_type": "military",
     "total_pages":  2, "total_chars":  1600, "total_words_approx":  345,
     "quality_score": 78},
    {"id": "const_2489", "year_th": 2489, "year_ce": 1946,
     "name_short": "รธน. 2489",
     "source_type": "image_pdf", "processing_method": "typhoon-ocr-1.5",
     "era": "early_democracy",   "regime_type": "civilian",
     "total_pages": 25, "total_chars": 33000, "total_words_approx": 7100,
     "quality_score": 87},
    {"id": "const_2490a", "year_th": 2490, "year_ce": 1947,
     "name_short": "รธน. 2490 (ชั่วคราว)",
     "source_type": "image_pdf", "processing_method": "typhoon-ocr-1.5",
     "era": "post_coup_1947",    "regime_type": "military",
     "total_pages":  8, "total_chars":  8500, "total_words_approx": 1850,
     "quality_score": 83},
    {"id": "const_2490b", "year_th": 2490, "year_ce": 1947,
     "name_short": "รธน. 2490 แก้ไข",
     "source_type": "image_pdf", "processing_method": "typhoon-ocr-1.5",
     "era": "post_coup_1947",    "regime_type": "military",
     "total_pages":  3, "total_chars":  2600, "total_words_approx":  570,
     "quality_score": 80},
    {"id": "const_2491a", "year_th": 2491, "year_ce": 1948,
     "name_short": "รธน. 2491 (ฉ.2)",
     "source_type": "image_pdf", "processing_method": "typhoon-ocr-1.5",
     "era": "post_coup_1947",    "regime_type": "military",
     "total_pages":  3, "total_chars":  2200, "total_words_approx":  480,
     "quality_score": 79},
    {"id": "const_2491b", "year_th": 2491, "year_ce": 1948,
     "name_short": "รธน. 2491 (ฉ.3)",
     "source_type": "image_pdf", "processing_method": "typhoon-ocr-1.5",
     "era": "post_coup_1947",    "regime_type": "military",
     "total_pages":  3, "total_chars":  2300, "total_words_approx":  495,
     "quality_score": 79},
    {"id": "const_2492", "year_th": 2492, "year_ce": 1949,
     "name_short": "รธน. 2492",
     "source_type": "image_pdf", "processing_method": "typhoon-ocr-1.5",
     "era": "post_coup_1947",    "regime_type": "civilian",
     "total_pages": 32, "total_chars": 44000, "total_words_approx": 9400,
     "quality_score": 88},
    {"id": "const_2495", "year_th": 2495, "year_ce": 1952,
     "name_short": "รธน. 2495",
     "source_type": "image_pdf", "processing_method": "typhoon-ocr-1.5",
     "era": "post_coup_1947",    "regime_type": "military",
     "total_pages": 20, "total_chars": 30000, "total_words_approx": 6400,
     "quality_score": 84},
    {"id": "const_2502", "year_th": 2502, "year_ce": 1959,
     "name_short": "ธรรมนูญ 2502",
     "source_type": "image_pdf", "processing_method": "typhoon-ocr-1.5",
     "era": "dictatorship",      "regime_type": "military",
     "total_pages":  7, "total_chars":  9000, "total_words_approx": 1950,
     "quality_score": 81},
    # ── Text PDFs (Text Extraction) ────────────────────────
    {"id": "const_2511", "year_th": 2511, "year_ce": 1968,
     "name_short": "รธน. 2511",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "dictatorship",      "regime_type": "military",
     "total_pages": 40, "total_chars": 65000, "total_words_approx": 13900,
     "quality_score": 94},
    {"id": "const_2515", "year_th": 2515, "year_ce": 1972,
     "name_short": "ธรรมนูญ 2515",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "dictatorship",      "regime_type": "military",
     "total_pages": 10, "total_chars": 11500, "total_words_approx": 2470,
     "quality_score": 93},
    {"id": "const_2517", "year_th": 2517, "year_ce": 1974,
     "name_short": "รธน. 2517",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "democratic_spring", "regime_type": "civilian",
     "total_pages": 65, "total_chars": 105000, "total_words_approx": 22500,
     "quality_score": 96},
    {"id": "const_2518", "year_th": 2518, "year_ce": 1975,
     "name_short": "รธน. 2518",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "democratic_spring", "regime_type": "civilian",
     "total_pages":  5, "total_chars":  3700, "total_words_approx":  800,
     "quality_score": 92},
    {"id": "const_2519", "year_th": 2519, "year_ce": 1976,
     "name_short": "รธน. 2519",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "democratic_spring", "regime_type": "military",
     "total_pages": 20, "total_chars": 23000, "total_words_approx": 4950,
     "quality_score": 93},
    {"id": "const_2520", "year_th": 2520, "year_ce": 1977,
     "name_short": "รธน. 2520",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "semi_democracy",    "regime_type": "military",
     "total_pages":  8, "total_chars":  7000, "total_words_approx": 1500,
     "quality_score": 91},
    {"id": "const_2521", "year_th": 2521, "year_ce": 1978,
     "name_short": "รธน. 2521",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "semi_democracy",    "regime_type": "semi_military",
     "total_pages": 55, "total_chars": 84000, "total_words_approx": 18000,
     "quality_score": 95},
    {"id": "const_2528", "year_th": 2528, "year_ce": 1985,
     "name_short": "รธน. 2528",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "semi_democracy",    "regime_type": "semi_military",
     "total_pages":  4, "total_chars":  3200, "total_words_approx":  690,
     "quality_score": 92},
    {"id": "const_2532", "year_th": 2532, "year_ce": 1989,
     "name_short": "รธน. 2532",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "semi_democracy",    "regime_type": "semi_military",
     "total_pages":  4, "total_chars":  3100, "total_words_approx":  665,
     "quality_score": 91},
    {"id": "const_2534a", "year_th": 2534, "year_ce": 1991,
     "name_short": "ธรรมนูญ 2534",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "modern_democracy",  "regime_type": "military",
     "total_pages":  5, "total_chars":  4200, "total_words_approx":  900,
     "quality_score": 93},
    {"id": "const_2534b", "year_th": 2534, "year_ce": 1991,
     "name_short": "รธน. 2534",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "modern_democracy",  "regime_type": "military",
     "total_pages": 50, "total_chars": 79000, "total_words_approx": 16900,
     "quality_score": 95},
    {"id": "const_2535a", "year_th": 2535, "year_ce": 1992,
     "name_short": "รธน. 2535 (ฉ.1)",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "modern_democracy",  "regime_type": "civilian",
     "total_pages":  4, "total_chars":  2800, "total_words_approx":  600,
     "quality_score": 92},
    {"id": "const_2535b", "year_th": 2535, "year_ce": 1992,
     "name_short": "รธน. 2535 (ฉ.2)",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "modern_democracy",  "regime_type": "civilian",
     "total_pages":  3, "total_chars":  2500, "total_words_approx":  540,
     "quality_score": 92},
    {"id": "const_2535c", "year_th": 2535, "year_ce": 1992,
     "name_short": "รธน. 2535 (ฉ.3)",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "modern_democracy",  "regime_type": "civilian",
     "total_pages":  3, "total_chars":  2200, "total_words_approx":  475,
     "quality_score": 90},
    {"id": "const_2535d", "year_th": 2535, "year_ce": 1992,
     "name_short": "รธน. 2535 (ฉ.4)",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "modern_democracy",  "regime_type": "civilian",
     "total_pages":  3, "total_chars":  2400, "total_words_approx":  515,
     "quality_score": 91},
    {"id": "const_2538", "year_th": 2538, "year_ce": 1995,
     "name_short": "รธน. 2538",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "modern_democracy",  "regime_type": "civilian",
     "total_pages":  4, "total_chars":  3000, "total_words_approx":  645,
     "quality_score": 93},
    {"id": "const_2539", "year_th": 2539, "year_ce": 1996,
     "name_short": "รธน. 2539",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "modern_democracy",  "regime_type": "civilian",
     "total_pages":  4, "total_chars":  3300, "total_words_approx":  710,
     "quality_score": 93},
    {"id": "const_2540", "year_th": 2540, "year_ce": 1997,
     "name_short": "รธน. 2540",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "modern_democracy",  "regime_type": "civilian",
     "total_pages": 72, "total_chars": 122000, "total_words_approx": 26100,
     "quality_score": 97},
    {"id": "const_2548", "year_th": 2548, "year_ce": 2005,
     "name_short": "รธน. 2548",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "modern_democracy",  "regime_type": "civilian",
     "total_pages":  4, "total_chars":  3400, "total_words_approx":  730,
     "quality_score": 94},
    {"id": "const_2549", "year_th": 2549, "year_ce": 2006,
     "name_short": "รธน. 2549 (ชั่วคราว)",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "post_coup_2006",    "regime_type": "military",
     "total_pages":  8, "total_chars":  9500, "total_words_approx": 2040,
     "quality_score": 95},
    {"id": "const_2550", "year_th": 2550, "year_ce": 2007,
     "name_short": "รธน. 2550",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "post_coup_2006",    "regime_type": "civilian",
     "total_pages": 75, "total_chars": 127000, "total_words_approx": 27200,
     "quality_score": 97},
    {"id": "const_2554a", "year_th": 2554, "year_ce": 2011,
     "name_short": "รธน. 2554 (ฉ.1)",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "post_coup_2006",    "regime_type": "civilian",
     "total_pages":  4, "total_chars":  3500, "total_words_approx":  750,
     "quality_score": 94},
    {"id": "const_2554b", "year_th": 2554, "year_ce": 2011,
     "name_short": "รธน. 2554 (ฉ.2)",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "post_coup_2006",    "regime_type": "civilian",
     "total_pages":  4, "total_chars":  3200, "total_words_approx":  685,
     "quality_score": 94},
    {"id": "const_2557", "year_th": 2557, "year_ce": 2014,
     "name_short": "รธน. 2557 (ชั่วคราว)",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "post_coup_2014",    "regime_type": "military",
     "total_pages": 10, "total_chars": 11800, "total_words_approx": 2530,
     "quality_score": 96},
    {"id": "const_2560", "year_th": 2560, "year_ce": 2017,
     "name_short": "รธน. 2560",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "post_coup_2014",    "regime_type": "semi_military",
     "total_pages": 82, "total_chars": 140000, "total_words_approx": 30000,
     "quality_score": 98},
    {"id": "const_2564", "year_th": 2564, "year_ce": 2021,
     "name_short": "รธน. 2564",
     "source_type": "text_pdf",  "processing_method": "pymupdf",
     "era": "post_coup_2014",    "regime_type": "semi_military",
     "total_pages":  4, "total_chars":  3700, "total_words_approx":  795,
     "quality_score": 96},
]

# ── โหลดข้อมูลจริง (ถ้ามี) หรือใช้ Mock ──────────────────
real_files = sorted(PROCESSED_DIR.glob('const_*.json'))

if real_files:
    print(f'✅ พบไฟล์ผลลัพธ์จริง {len(real_files)} ไฟล์ — ใช้ข้อมูลจริง')
    rows = []
    for f in real_files:
        with open(f, encoding='utf-8') as fp:
            d = json.load(fp)
        rows.append({
            'id':                 d.get('id'),
            'year_th':            d.get('year_th'),
            'year_ce':            d.get('year_ce'),
            'name_short':         d.get('name_short', d.get('id')),
            'source_type':        d.get('source_type'),
            'processing_method':  d.get('processing_method'),
            'era':                d.get('era'),
            'regime_type':        d.get('regime_type'),
            'total_pages':        d.get('total_pages', 0),
            'total_chars':        d.get('metadata', {}).get('total_chars', 0),
            'total_words_approx': d.get('metadata', {}).get('total_words_approx', 0),
            'quality_score':      None,
        })
    # โหลด QA score ถ้ามี
    qa_path = PROCESSED_DIR / 'qa_report.csv'
    if qa_path.exists():
        qa_df = pd.read_csv(qa_path)
        qa_map = dict(zip(qa_df['id'], qa_df['quality_score']))
        for r in rows:
            r['quality_score'] = qa_map.get(r['id'], 0)
    df = pd.DataFrame(rows)
    DATA_SOURCE = 'real'
else:
    print('⚠️  ยังไม่พบไฟล์ผลลัพธ์ — ใช้ Mock Data แทน')
    print('   (รัน run_pipeline.py เพื่อประมวลผลจริง)')
    df = pd.DataFrame(MOCK_DATA)
    DATA_SOURCE = 'mock'

df = df.sort_values('year_th').reset_index(drop=True)
print(f'\n📊 โหลดข้อมูล {len(df)} รายการ  [source: {DATA_SOURCE}]')


---
## 📋 ส่วนที่ 1 — Pipeline Status
ตรวจสอบว่ารัฐธรรมนูญแต่ละฉบับถูกประมวลผลไปแล้วหรือยัง


In [ ]:
# ── Pipeline Status ────────────────────────────────────────
processed_ids = {f.stem for f in PROCESSED_DIR.glob('const_*.json')}
all_ids       = [d['id'] for d in MOCK_DATA]  # ใช้ MOCK_DATA เพื่อรู้ทุก ID

status_rows = []
for d in MOCK_DATA:
    cid    = d['id']
    done   = cid in processed_ids
    status_rows.append({
        'ปี พ.ศ.':          d['year_th'],
        'ชื่อย่อ':          d['name_short'],
        'ประเภท':           '🖼️ Image PDF' if d['source_type'] == 'image_pdf' else '📄 Text PDF',
        'วิธีประมวลผล':    'Typhoon OCR 1.5' if d['source_type'] == 'image_pdf' else 'PyMuPDF',
        'สถานะ':            '✅ เสร็จสิ้น' if done else '⏳ รอดำเนินการ',
    })

status_df = pd.DataFrame(status_rows)

# ── Style ─────────────────────────────────────────────────
def color_status(val):
    if '✅' in str(val):
        return 'background-color: #e8f5e9; color: #2e7d32; font-weight: bold'
    elif '⏳' in str(val):
        return 'background-color: #fff8e1; color: #f57f17'
    return ''

styled = (status_df.style
          .applymap(color_status, subset=['สถานะ'])
          .set_table_styles([{'selector': 'th',
                              'props': [('background-color', '#37474f'),
                                        ('color', 'white'),
                                        ('font-weight', 'bold'),
                                        ('padding', '6px 12px')]}])
          .set_properties(**{'text-align': 'left', 'padding': '5px 10px'})
          .hide(axis='index'))

done_count  = sum(1 for d in MOCK_DATA if d['id'] in processed_ids)
total_count = len(MOCK_DATA)

display(HTML(f"""
<div style='margin-bottom:10px; padding:10px; background:#f5f5f5; border-radius:8px;'>
  <b>สรุป:</b>
  เสร็จสิ้น <b style='color:#2e7d32'>{done_count}</b> /
  ทั้งหมด <b>{total_count}</b> ฉบับ
  &nbsp;|&nbsp;
  Image PDF: <b>12</b> ฉบับ &nbsp;|&nbsp;
  Text PDF: <b>26</b> ฉบับ
</div>
"""))
display(styled)


---
## 📊 ส่วนที่ 2 — Dataset Overview
ภาพรวมของชุดข้อมูลที่ได้จาก Pipeline


In [ ]:
# ── Dataset Overview Statistics ────────────────────────────
total_docs   = len(df)
total_pages  = df['total_pages'].sum()
total_words  = df['total_words_approx'].sum()
total_chars  = df['total_chars'].sum()
avg_words    = df['total_words_approx'].mean()
year_min     = df['year_th'].min()
year_max     = df['year_th'].max()

img_count    = len(df[df['source_type'] == 'image_pdf'])
txt_count    = len(df[df['source_type'] == 'text_pdf'])

display(HTML(f"""
<h4 style='color:#37474f'>สถิติภาพรวมของ Corpus</h4>
<table style='border-collapse:collapse; width:100%; font-size:13px;'>
  <tr style='background:#37474f; color:white;'>
    <th style='padding:8px 14px; text-align:left;'>ตัวชี้วัด</th>
    <th style='padding:8px 14px; text-align:right;'>ค่า</th>
  </tr>
  <tr style='background:#f5f5f5;'>
    <td style='padding:7px 14px;'>จำนวนเอกสารทั้งหมด</td>
    <td style='padding:7px 14px; text-align:right; font-weight:bold;'>{total_docs} ฉบับ</td>
  </tr>
  <tr>
    <td style='padding:7px 14px;'>  ├ Image PDF (ต้องทำ OCR)</td>
    <td style='padding:7px 14px; text-align:right;'>{img_count} ฉบับ (พ.ศ. 2475–2502)</td>
  </tr>
  <tr style='background:#f5f5f5;'>
    <td style='padding:7px 14px;'>  └ Text PDF (ดึงข้อความโดยตรง)</td>
    <td style='padding:7px 14px; text-align:right;'>{txt_count} ฉบับ (พ.ศ. 2511–2564)</td>
  </tr>
  <tr>
    <td style='padding:7px 14px;'>ช่วงปีที่ครอบคลุม</td>
    <td style='padding:7px 14px; text-align:right;'>พ.ศ. {year_min} – {year_max} ({year_max - year_min} ปี)</td>
  </tr>
  <tr style='background:#f5f5f5;'>
    <td style='padding:7px 14px;'>จำนวนหน้ารวมทั้งหมด</td>
    <td style='padding:7px 14px; text-align:right; font-weight:bold;'>{total_pages:,} หน้า</td>
  </tr>
  <tr>
    <td style='padding:7px 14px;'>จำนวนคำโดยประมาณ (รวม)</td>
    <td style='padding:7px 14px; text-align:right; font-weight:bold;'>{total_words:,.0f} คำ</td>
  </tr>
  <tr style='background:#f5f5f5;'>
    <td style='padding:7px 14px;'>จำนวนตัวอักษร (รวม)</td>
    <td style='padding:7px 14px; text-align:right;'>{total_chars:,} ตัวอักษร</td>
  </tr>
  <tr>
    <td style='padding:7px 14px;'>จำนวนคำเฉลี่ยต่อฉบับ</td>
    <td style='padding:7px 14px; text-align:right;'>{avg_words:,.0f} คำ</td>
  </tr>
</table>
"""))

# ── Top 5 ฉบับยาวที่สุด ────────────────────────────────────
print('\n📌 5 อันดับรัฐธรรมนูญที่มีจำนวนคำมากที่สุด:')
top5 = df.nlargest(5, 'total_words_approx')[['name_short', 'year_th', 'total_words_approx', 'total_pages']]
top5.columns = ['ชื่อย่อ', 'ปี พ.ศ.', 'จำนวนคำ (ประมาณ)', 'จำนวนหน้า']
top5 = top5.reset_index(drop=True)
top5.index = top5.index + 1
display(top5.style.format({'จำนวนคำ (ประมาณ)': '{:,.0f}'})
               .background_gradient(subset=['จำนวนคำ (ประมาณ)'], cmap='YlOrRd')
               .set_properties(**{'text-align': 'left', 'padding': '5px 10px'}))


---
## 🥧 ส่วนที่ 3 — การกระจายประเภทเอกสารและวิธีประมวลผล


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('การกระจายประเภทเอกสารและวิธีการประมวลผล',
             fontsize=14, fontweight='bold', y=1.01)

# ── แผนภูมิที่ 1: Image PDF vs Text PDF ───────────────────
ax1 = axes[0]
src_counts = df['source_type'].value_counts()
labels1 = ['Image PDF\n(OCR — 2475–2502)', 'Text PDF\n(Extract — 2511–2564)']
colors1 = ['#FF7043', '#42A5F5']
wedges, texts, autotexts = ax1.pie(
    src_counts.values,
    labels=labels1,
    colors=colors1,
    autopct='%1.0f%%',
    startangle=90,
    pctdistance=0.75,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
)
for at in autotexts:
    at.set_fontsize(12)
    at.set_fontweight('bold')
    at.set_color('white')
ax1.set_title(f'ประเภทเอกสาร (n={len(df)})', fontsize=12, pad=12)

# ── แผนภูมิที่ 2: จำนวนคำ Image vs Text ───────────────────
ax2 = axes[1]
word_by_type = df.groupby('source_type')['total_words_approx'].sum()
bar_labels   = ['Image PDF\n(OCR)', 'Text PDF\n(Extract)']
bar_values   = [
    word_by_type.get('image_pdf', 0),
    word_by_type.get('text_pdf',  0),
]
bars = ax2.bar(bar_labels, bar_values, color=colors1, edgecolor='white',
               linewidth=1.5, width=0.5)
for bar, val in zip(bars, bar_values):
    ax2.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 500,
             f'{val:,.0f}\nคำ',
             ha='center', va='bottom', fontsize=11, fontweight='bold')
ax2.set_title('จำนวนคำรวมแยกตามประเภทเอกสาร', fontsize=12, pad=12)
ax2.set_ylabel('จำนวนคำ (ประมาณ)')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax2.set_ylim(0, max(bar_values) * 1.2)

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_source_distribution.png',
            bbox_inches='tight', dpi=150)
plt.show()
print('💾 บันทึกรูปที่ data/processed/fig_source_distribution.png')


---
## 📏 ส่วนที่ 4 — จำนวนคำในรัฐธรรมนูญแต่ละฉบับ


In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

# เรียงตามปีพ.ศ.
plot_df = df.sort_values('year_th').reset_index(drop=True)

colors = [ERA_COLORS.get(era, '#90A4AE') for era in plot_df['era']]

bars = ax.barh(
    y=plot_df['name_short'],
    width=plot_df['total_words_approx'],
    color=colors,
    edgecolor='white',
    linewidth=0.5,
    height=0.75,
)

# ป้ายบอกค่า
for bar, val in zip(bars, plot_df['total_words_approx']):
    if val > 1500:
        ax.text(val + 200, bar.get_y() + bar.get_height() / 2,
                f'{val:,.0f}',
                va='center', ha='left', fontsize=8.5, color='#37474f')

# Legend (ยุคสมัย)
unique_eras  = plot_df['era'].unique()
legend_patches = [
    mpatches.Patch(color=ERA_COLORS.get(e, '#90A4AE'),
                   label=ERA_LABELS_TH.get(e, e))
    for e in [
        'early_democracy', 'post_coup_1947', 'dictatorship',
        'democratic_spring', 'semi_democracy', 'modern_democracy',
        'post_coup_2006', 'post_coup_2014',
    ]
    if e in unique_eras
]
ax.legend(handles=legend_patches, title='ยุคสมัยทางการเมือง',
          bbox_to_anchor=(1.01, 1), loc='upper left',
          fontsize=9, title_fontsize=10)

ax.set_xlabel('จำนวนคำโดยประมาณ', fontsize=12)
ax.set_title('จำนวนคำในรัฐธรรมนูญไทยแต่ละฉบับ (เรียงตามปีพ.ศ.)',
             fontsize=13, fontweight='bold', pad=14)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_word_count_per_constitution.png',
            bbox_inches='tight', dpi=150)
plt.show()
print('💾 บันทึกรูปที่ data/processed/fig_word_count_per_constitution.png')


---
## 📈 ส่วนที่ 5 — Word Count Timeline (แนวโน้มตามเวลา)


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

plot_df = df.sort_values('year_th').reset_index(drop=True)

# แยกสีตาม source_type
for idx, row in plot_df.iterrows():
    color = '#FF7043' if row['source_type'] == 'image_pdf' else '#42A5F5'
    ax.scatter(row['year_th'], row['total_words_approx'],
               color=color, s=60, zorder=5)

# เส้นเชื่อม
ax.plot(plot_df['year_th'], plot_df['total_words_approx'],
        color='#90A4AE', linewidth=1.2, zorder=3, alpha=0.6)

# Highlight ฉบับสำคัญ
highlights = {
    2475: 'รธน. 2475\n(ฉบับแรก)',
    2517: 'รธน. 2517\n(ยาวที่สุดยุคแรก)',
    2540: 'รธน. 2540\n(ฉบับประชาชน)',
    2560: 'รธน. 2560\n(ปัจจุบัน)',
}
for year_th, label in highlights.items():
    row = plot_df[plot_df['year_th'] == year_th]
    if not row.empty:
        y_val = row['total_words_approx'].values[0]
        ax.annotate(label,
                    xy=(year_th, y_val),
                    xytext=(year_th + 1, y_val + 1500),
                    fontsize=8.5,
                    arrowprops={'arrowstyle': '->', 'color': '#37474f', 'lw': 1},
                    color='#37474f')

# แบ่งพื้นที่ยุคสมัยด้วยสีพื้นหลัง
era_ranges = [
    (2475, 2490, 'early_democracy'),
    (2490, 2503, 'post_coup_1947'),
    (2503, 2517, 'dictatorship'),
    (2517, 2520, 'democratic_spring'),
    (2520, 2534, 'semi_democracy'),
    (2534, 2549, 'modern_democracy'),
    (2549, 2558, 'post_coup_2006'),
    (2558, 2566, 'post_coup_2014'),
]
ymax = plot_df['total_words_approx'].max() * 1.15
for start, end, era in era_ranges:
    ax.axvspan(start, end,
               alpha=0.08,
               color=ERA_COLORS.get(era, '#90A4AE'),
               zorder=1)

# Legend
ocr_patch  = mpatches.Patch(color='#FF7043', label='Image PDF (OCR)')
text_patch = mpatches.Patch(color='#42A5F5', label='Text PDF (Extract)')
ax.legend(handles=[ocr_patch, text_patch], loc='upper left', fontsize=9)

ax.set_xlabel('ปี พ.ศ.', fontsize=11)
ax.set_ylabel('จำนวนคำโดยประมาณ', fontsize=11)
ax.set_title('แนวโน้มจำนวนคำในรัฐธรรมนูญไทย พ.ศ. 2475–2564',
             fontsize=13, fontweight='bold', pad=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_xlim(2470, 2570)
ax.set_ylim(0, ymax)

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_wordcount_timeline.png',
            bbox_inches='tight', dpi=150)
plt.show()
print('💾 บันทึกรูปที่ data/processed/fig_wordcount_timeline.png')


---
## 🏛️ ส่วนที่ 6 — วิเคราะห์ตามยุคสมัยทางการเมือง


In [ ]:
# ── สถิติรายยุค ───────────────────────────────────────────
era_order = [
    'early_democracy', 'post_coup_1947', 'dictatorship',
    'democratic_spring', 'semi_democracy', 'modern_democracy',
    'post_coup_2006', 'post_coup_2014',
]

era_stats = (df.groupby('era')
               .agg(
                   จำนวนฉบับ=('id',                  'count'),
                   จำนวนคำรวม=('total_words_approx', 'sum'),
                   จำนวนคำเฉลี่ย=('total_words_approx', 'mean'),
                   จำนวนหน้ารวม=('total_pages',        'sum'),
               )
               .reindex([e for e in era_order if e in df['era'].unique()])
               .reset_index())

era_stats['ยุคสมัย'] = era_stats['era'].map(ERA_LABELS_TH)
era_stats_display = era_stats[['ยุคสมัย', 'จำนวนฉบับ',
                                'จำนวนคำรวม', 'จำนวนคำเฉลี่ย',
                                'จำนวนหน้ารวม']].copy()

display(era_stats_display.style
    .format({'จำนวนคำรวม': '{:,.0f}', 'จำนวนคำเฉลี่ย': '{:,.0f}'})
    .background_gradient(subset=['จำนวนคำรวม'],    cmap='Blues')
    .background_gradient(subset=['จำนวนฉบับ'],     cmap='Greens')
    .set_properties(**{'text-align': 'left', 'padding': '6px 12px'})
    .set_table_styles([{'selector': 'th',
                        'props': [('background-color', '#37474f'),
                                  ('color', 'white'),
                                  ('padding', '7px 12px')]}])
    .hide(axis='index'))

# ── กราฟ: จำนวนฉบับและคำรวมต่อยุค ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('วิเคราะห์รัฐธรรมนูญตามยุคสมัยทางการเมือง',
             fontsize=13, fontweight='bold')

era_colors_list = [ERA_COLORS.get(e, '#90A4AE') for e in era_stats['era']]
short_labels    = [ERA_LABELS_TH.get(e, e).split('(')[0].strip()
                   for e in era_stats['era']]

# กราฟซ้าย: จำนวนฉบับ
axes[0].bar(short_labels, era_stats['จำนวนฉบับ'],
            color=era_colors_list, edgecolor='white', linewidth=1)
axes[0].set_title('จำนวนรัฐธรรมนูญต่อยุค')
axes[0].set_ylabel('จำนวนฉบับ')
axes[0].tick_params(axis='x', rotation=35, labelsize=8)

# กราฟขวา: จำนวนคำรวม
axes[1].bar(short_labels, era_stats['จำนวนคำรวม'],
            color=era_colors_list, edgecolor='white', linewidth=1)
axes[1].set_title('จำนวนคำรวมต่อยุค')
axes[1].set_ylabel('จำนวนคำ')
axes[1].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
axes[1].tick_params(axis='x', rotation=35, labelsize=8)

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_by_era.png',
            bbox_inches='tight', dpi=150)
plt.show()
print('💾 บันทึกรูปที่ data/processed/fig_by_era.png')


---
## ✅ ส่วนที่ 7 — QA Report: คุณภาพข้อมูลที่ได้

คะแนนคุณภาพ (0–100) ที่คำนวณจาก:
- สัดส่วนอักษรไทยในเอกสาร (ควรมากกว่า 30%)
- จำนวนคำสำคัญที่พบ (`มาตรา`, `รัฐธรรมนูญ`)
- ความถี่ OCR error patterns
- จำนวนหน้าว่างเปล่า


In [ ]:
qa_df = df[df['quality_score'].notna()].copy()

if qa_df.empty:
    print('⚠️  ยังไม่มีข้อมูล QA — รัน 03_validate_output.py ก่อน')
else:
    fig, ax = plt.subplots(figsize=(14, 8))

    qa_sorted  = qa_df.sort_values('year_th')
    bar_colors = ['#4CAF50' if s >= 90
                  else '#FFC107' if s >= 75
                  else '#f44336'
                  for s in qa_sorted['quality_score']]

    bars = ax.barh(
        qa_sorted['name_short'],
        qa_sorted['quality_score'],
        color=bar_colors,
        edgecolor='white',
        linewidth=0.5,
        height=0.75,
    )

    for bar, val, src in zip(bars,
                              qa_sorted['quality_score'],
                              qa_sorted['source_type']):
        label = f'{val:.0f}  [{"OCR" if src == "image_pdf" else "TXT"}]'
        ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2,
                label, va='center', fontsize=8.5)

    ax.axvline(x=90, color='#4CAF50', linestyle='--',
               linewidth=1, alpha=0.7, label='เกณฑ์ดี (≥90)')
    ax.axvline(x=75, color='#FFC107', linestyle='--',
               linewidth=1, alpha=0.7, label='เกณฑ์พอใช้ (≥75)')

    ax.set_xlim(0, 105)
    ax.set_xlabel('Quality Score (0–100)', fontsize=11)
    ax.set_title('คุณภาพข้อมูลรัฐธรรมนูญแต่ละฉบับ (QA Score)',
                 fontsize=13, fontweight='bold', pad=12)
    ax.tick_params(axis='y', labelsize=9)
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(PROCESSED_DIR / 'fig_qa_scores.png',
                bbox_inches='tight', dpi=150)
    plt.show()

    # สรุป
    scores = qa_df['quality_score']
    print(f'\n📊 สรุป QA Score:')
    print(f'   เฉลี่ย     : {scores.mean():.1f} / 100')
    print(f'   สูงสุด     : {scores.max():.1f} ({qa_df.loc[scores.idxmax(), "name_short"]})')
    print(f'   ต่ำสุด     : {scores.min():.1f} ({qa_df.loc[scores.idxmin(), "name_short"]})')
    print(f'   Score ≥ 90 : {(scores >= 90).sum()} ฉบับ  ✅')
    print(f'   Score 75–89: {((scores >= 75) & (scores < 90)).sum()} ฉบับ  ⚠️')
    print(f'   Score < 75 : {(scores < 75).sum()} ฉบับ  ❌')
    print('💾 บันทึกรูปที่ data/processed/fig_qa_scores.png')


---
## 📖 ส่วนที่ 8 — Sample Text Preview
แสดงตัวอย่างข้อความที่ได้จาก Pipeline


In [ ]:
# ── ตัวอย่างข้อความจริง (ถ้ามี) หรือ Sample text ──────────
SAMPLE_TEXTS = {
    'const_2475': (
        'มาตรา ๑ อำนาจอธิปไตยสูงสุดของประเทศนั้นเป็นของราษฎรทั้งหลาย\n'
        'มาตรา ๒ ให้ใช้รัฐธรรมนูญนี้แทนธรรมนูญการปกครองแผ่นดินสยามชั่วคราว\n'
        'พุทธศักราช ๒๔๗๕\n'
        'มาตรา ๓ ประเทศสยามเป็นราชอาณาจักรอันหนึ่งอันเดียว จะแบ่งแยกมิได้\n'
        'มาตรา ๔ พระมหากษัตริย์ทรงเป็นประมุขสูงสุดแห่งรัฐ'
    ),
    'const_2540': (
        'มาตรา ๑ ประเทศไทยเป็นราชอาณาจักรอันหนึ่งอันเดียวจะแบ่งแยกมิได้\n'
        'มาตรา ๒ ประเทศไทยมีการปกครองระบอบประชาธิปไตยอันมีพระมหากษัตริย์\n'
        'ทรงเป็นประมุข\n'
        'มาตรา ๓ อำนาจอธิปไตยเป็นของปวงชนชาวไทย พระมหากษัตริย์ผู้ทรงเป็น\n'
        'ประมุขทรงใช้อำนาจนั้นทางรัฐสภา คณะรัฐมนตรี และศาล'
    ),
    'const_2560': (
        'มาตรา ๑ ประเทศไทยเป็นราชอาณาจักรอันหนึ่งอันเดียวจะแบ่งแยกมิได้\n'
        'มาตรา ๒ ประเทศไทยมีการปกครองระบอบประชาธิปไตยอันมีพระมหากษัตริย์\n'
        'ทรงเป็นประมุข\n'
        'มาตรา ๓ อำนาจอธิปไตยเป็นของปวงชนชาวไทย พระมหากษัตริย์ผู้ทรงเป็น\n'
        'ประมุขทรงใช้อำนาจนั้นทางรัฐสภา คณะรัฐมนตรี และศาลตามบทบัญญัติ\n'
        'แห่งรัฐธรรมนูญ'
    ),
}

# เลือกฉบับที่จะแสดง
preview_ids = ['const_2475', 'const_2540', 'const_2560']

for pid in preview_ids:
    meta = df[df['id'] == pid]
    if meta.empty:
        continue
    row = meta.iloc[0]

    # ลองโหลดข้อความจริง
    json_path = PROCESSED_DIR / f'{pid}.json'
    if json_path.exists():
        with open(json_path, encoding='utf-8') as f:
            real_data = json.load(f)
        text_preview = real_data.get('full_text', '')[:500]
        source_label = '📄 ข้อความจริงจาก Pipeline'
    else:
        text_preview = SAMPLE_TEXTS.get(pid, '(ไม่มีตัวอย่าง)')
        source_label = '📝 ตัวอย่าง (Mock)'

    src_icon  = '🖼️' if row['source_type'] == 'image_pdf' else '📄'
    src_label = 'Image PDF → Typhoon OCR 1.5' if row['source_type'] == 'image_pdf' \
                else 'Text PDF → PyMuPDF'

    display(HTML(f"""
    <div style='border:1px solid #e0e0e0; border-radius:10px;
                padding:16px; margin:12px 0;
                background: #fafafa;'>
      <div style='display:flex; justify-content:space-between;
                  align-items:center; margin-bottom:10px;'>
        <h4 style='margin:0; color:#1565C0;'>
          {src_icon} {row['name_short']}
          <span style='font-size:12px; color:#666;
                       font-weight:normal; margin-left:8px;'>
            (พ.ศ. {row['year_th']})
          </span>
        </h4>
        <span style='font-size:11px; color:#888;'>{source_label}</span>
      </div>
      <div style='font-size:11px; color:#555; margin-bottom:8px;'>
        🔧 วิธีประมวลผล: <b>{src_label}</b>
        &nbsp;|&nbsp;
        📃 {int(row['total_pages'])} หน้า
        &nbsp;|&nbsp;
        💬 ~{int(row['total_words_approx']):,} คำ
      </div>
      <pre style='background:#fff; border:1px solid #e0e0e0;
                  border-radius:6px; padding:12px;
                  font-family: Sarabun, Tahoma, sans-serif;
                  font-size:13px; line-height:1.7;
                  white-space:pre-wrap; color:#212121;
                  max-height:180px; overflow-y:auto;'>{text_preview}</pre>
    </div>
    """))


---
## 📁 ส่วนที่ 9 — Output Files Summary
ไฟล์ผลลัพธ์ทั้งหมดที่ได้จาก Pipeline


In [ ]:
# ── รายการไฟล์ใน data/processed/ ─────────────────────────
output_files = sorted(PROCESSED_DIR.glob('*'))

if not output_files:
    print('⚠️  ยังไม่มีไฟล์ผลลัพธ์ใน data/processed/')
    print('   รัน python run_pipeline.py เพื่อเริ่มต้น Pipeline')
else:
    file_rows = []
    for f in output_files:
        if f.is_file():
            size_kb = f.stat().st_size / 1024
            ext     = f.suffix.lower()
            icon    = {'json': '📋', 'csv': '📊', 'txt': '📝',
                       'png': '🖼️', 'log': '📜'}.get(ext.lstrip('.'), '📄')
            file_rows.append({
                '':        icon,
                'ไฟล์':   f.name,
                'ประเภท': ext.upper().lstrip('.'),
                'ขนาด':   f'{size_kb:.1f} KB',
            })

    files_df = pd.DataFrame(file_rows)
    display(files_df.style
        .set_properties(**{'text-align': 'left', 'padding': '5px 10px'})
        .set_table_styles([{'selector': 'th',
                            'props': [('background-color', '#37474f'),
                                      ('color', 'white'),
                                      ('padding', '6px 12px')]}])
        .hide(axis='index'))

    # สรุปขนาดรวม
    total_kb = sum(f.stat().st_size for f in output_files if f.is_file()) / 1024
    json_count = len(list(PROCESSED_DIR.glob('const_*.json')))
    print(f'\n📦 ขนาดรวมทั้งหมด : {total_kb:.1f} KB ({total_kb/1024:.2f} MB)')
    print(f'📋 ไฟล์ JSON รายฉบับ : {json_count} ไฟล์')
    print(f'📊 พร้อมส่งต่อให้ EDA และ ML Modeling')


---
## 🎯 สรุปและขั้นตอนถัดไป

### ✅ สิ่งที่ทำสำเร็จใน Data Preparation

| ขั้นตอน | สิ่งที่ทำ | เครื่องมือ |
|---------|----------|----------|
| **PDF Classification** | แบ่งเอกสาร 38 ฉบับ → Image PDF / Text PDF | `config.py` |
| **OCR (Image PDF)** | แปลงรัฐธรรมนูญ 12 ฉบับ (2475–2502) เป็นข้อความ | **Typhoon OCR 1.5** |
| **Text Extraction** | ดึงข้อความจากรัฐธรรมนูญ 26 ฉบับ (2511–2564) | **PyMuPDF + pdfplumber** |
| **Post-Processing** | Normalize Unicode, ลบ Header/Footer, แปลงเลขไทย | `pythainlp` |
| **Validation** | ตรวจสอบคุณภาพ QA Score (0–100) | `03_validate_output.py` |
| **Structured Output** | บันทึก JSON + CSV + Plain Text | `run_pipeline.py` |

---

### ➡️ ขั้นตอนถัดไป (งานของสมาชิกคนอื่น)

```
Data Preparation (✅ เสร็จแล้ว)
       │
       ▼
EDA — Exploratory Data Analysis
  ▸ Dataset Overview (จำนวนมาตรา, ความยาว)
  ▸ Text Analysis (keyword frequency, distribution)
       │
       ▼
Data Cleaning & Preprocessing
  ▸ Tokenization (PyThaiNLP / Attacut)
  ▸ Stopword Removal
       │
       ▼
Feature Extraction
  ▸ Bag-of-Words, TF-IDF
  ▸ WangChanBERTa Embeddings
       │
       ▼
ML Modeling
  ▸ Topic Modeling (LDA / NMF)
  ▸ Constitution Similarity Matrix
```

---

### 📚 แหล่งข้อมูลที่เกี่ยวข้อง

- **Typhoon OCR 1.5**: https://docs.opentyphoon.ai/en/ocr/
- **รัฐธรรมนูญไทย**: https://catalog.parliament.go.th/dataset/12_01
- **PyMuPDF**: https://pymupdf.readthedocs.io/
- **PyThaiNLP**: https://pythainlp.github.io/

---
*CPE232 Data Models — Final Project — กลุ่ม Constitution*
